In [1]:
# Cell 1: Imports and Path Setup
import sys
from pathlib import Path
import json
import os
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, asdict, field
import pickle

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# VectorBT Data Preprocessor
from crypto_analysis.vectorbt_optimizer import VectorBTDataPreprocessor

In [3]:
preprocessor = VectorBTDataPreprocessor(
    remove_raw_indicators=False,
    target_shift=1,
    sequence_length=16,
    stride=1,
    scaler_type='minmax',
    target_column='tradeable',
    normalize_by_close=False,
)

doge_df = preprocessor.load_csv("notebooks/test_csv/DOGE_optimized.csv")
feats = json.loads(open("notebooks/doge_feats.json").read())
print("len features", len(feats))
doge_df = doge_df[["tradeable"]+feats]
preprocessor.fit_transform(doge_df)
print(f"\nAfter fitting:")
print(preprocessor)
print(f"\nFeature columns: {preprocessor.get_feature_names()}")
print(f"Number of features: {preprocessor.get_num_features()}")

# Store for later use
n_features = preprocessor.get_num_features()
feature_names = preprocessor.get_feature_names()

len features 146

After fitting:
VectorBTDataPreprocessor(sequence_length=16, target_shift=1, stride=1, remove_raw_indicators=False, normalize_by_close=False, n_features=139, n_scaled=58, n_binary=81, scaler_type='minmax')

Feature columns: ['open', 'low', 'close', 'volume', 'RSI_entry', 'STOCH_entry', 'STOCH_slowd', 'STOCHRSI_entry', 'STOCHRSI_exit', 'STOCHRSI_fastd', 'CCI_entry', 'CCI_exit', 'CCI_cci', 'MFI_entry', 'MFI_exit', 'MFI_mfi', 'WILLR_entry', 'WILLR_exit', 'WILLR_willr', 'CMO_entry', 'CMO_exit', 'CMO_cmo', 'ADX_entry', 'ADX_adx', 'MOM_exit', 'MOM_mom', 'ROC_entry', 'TRIX_entry', 'TRIX_trix', 'ULTOSC_entry', 'ULTOSC_exit', 'ULTOSC_ultosc', 'APO_exit', 'PPO_exit', 'PPO_ppo', 'BOP_entry', 'BOP_exit', 'BOP_bop', 'AROON_entry', 'AROON_aroondown', 'AROONOSC_entry', 'AROONOSC_aroonosc', 'ADXR_entry', 'ADXR_exit', 'ADXR_adxr', 'DX_entry', 'DX_exit', 'MACDEXT_entry', 'MACDEXT_exit', 'MACDEXT_macd', 'MACDEXT_macdsignal', 'MACDFIX_entry', 'MACDFIX_exit', 'MACDFIX_macd', 'MACDFIX_macds

In [4]:
features, targets = preprocessor.fit_transform(doge_df)

In [5]:
feat_df = pd.DataFrame(features, columns=feature_names)
feat_df["targets"] = targets
feat_df

,open,low,close,volume,RSI_entry,STOCH_entry,STOCH_slowd,STOCHRSI_entry,STOCHRSI_exit,STOCHRSI_fastd,...,HT_PHASOR_entry,HT_PHASOR_exit,HT_PHASOR_inphase,HT_PHASOR_quadrature,HT_SINE_entry,HT_SINE_leadsine,HT_TRENDMODE_entry,HT_TRENDMODE_exit,HT_TRENDMODE_ht_trendmode,targets
0,0.705271,0.741141,0.711574,0.014779,0.0,0.0,0.668543,0.0,0.0,0.552839,...,1.0,0.0,0.536234,0.426221,0.0,0.998958,0.0,0.0,0.0,1
1,0.711588,0.743188,0.706438,0.013981,0.0,0.0,0.531045,0.0,0.0,0.354351,...,0.0,0.0,0.513379,0.394064,1.0,0.956417,0.0,0.0,0.0,0
2,0.706419,0.737964,0.700727,0.013946,0.0,0.0,0.375731,1.0,0.0,0.249700,...,0.0,0.0,0.490793,0.381666,0.0,0.903264,0.0,0.0,0.0,1
3,0.700740,0.718842,0.684617,0.051238,0.0,0.0,0.245463,1.0,0.0,0.050253,...,0.0,0.0,0.456097,0.376930,0.0,0.660159,1.0,0.0,1.0,1
4,0.684597,0.720338,0.690104,0.038538,0.0,0.0,0.172765,0.0,0.0,0.092051,...,0.0,0.0,0.426398,0.396221,0.0,0.173281,0.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9000,0.032446,0.097141,0.031998,0.019877,0.0,0.0,0.354353,0.0,0.0,0.114135,...,0.0,0.0,0.416685,0.356954,0.0,0.509751,0.0,0.0,1.0,0
9001,0.031936,0.097599,0.031998,0.018458,0.0,0.0,0.396128,0.0,0.0,0.123243,...,0.0,0.0,0.364484,0.372779,0.0,0.540200,0.0,0.0,1.0,0
9002,0.031968,0.096224,0.030881,0.018283,0.0,0.0,0.423370,0.0,0.0,0.344157,...,0.0,1.0,0.354561,0.435598,0.0,0.581454,0.0,0.0,1.0,0
9003,0.030851,0.097263,0.032030,0.008565,0.0,0.0,0.476692,0.0,1.0,0.637091,...,0.0,0.0,0.378590,0.474839,0.0,0.617260,0.0,0.0,1.0,0
